# Multi-Agent Development Workflow with Claude Code

## Introduction

What if your AI coding assistant could enforce the same quality process that a well-run engineering team follows — design review, security review, implementation, testing — automatically, every time?

This cookbook demonstrates a **multi-agent development workflow** using Claude Code's native filesystem architecture. Instead of a single AI writing all the code, specialized agents collaborate through an orchestrator with enforced quality gates.

### The Pattern

```
User: /implement "Add due dates to todos"
  ↓
Coordinator (orchestrator)
  → Architect: produces design document
  → Security: reviews design for vulnerabilities  
  → Developer: implements from approved design
  → Security: reviews code for vulnerabilities
  → QA: writes tests, runs full suite
  ↓
Result: feature implemented with design doc, security clearance, and tests
```

### Why Multi-Agent?

A single agent writing code tends to:
- Skip security review (it wrote the code, so it trusts the code)
- Write tests that mirror the implementation (same blind spots)
- Design and implement simultaneously (no separation of concerns)

Specialized agents with **role boundaries** solve this:
- The security agent has never seen the implementation rationale — it reviews cold
- The QA agent writes tests from the *design doc*, not from reading the code
- The architect designs without being biased by what is easy to implement

### What You'll Learn

1. How to define specialist agents using `.claude/agents/*.md` files
2. How to build workflow commands with `.claude/commands/*.md`
3. How to enforce quality gates through orchestrator logic
4. How to handle failures and re-entry loops
5. How to persist workflow state for cross-session resumability

### Prerequisites

- Claude Code CLI installed (`npm install -g @anthropic-ai/claude-code`)
- An Anthropic API key
- Basic familiarity with Claude Code slash commands

## Architecture

### Project Structure

```
your-project/
├── .claude/
│   ├── CLAUDE.md              # Project rules + workflow documentation
│   ├── agents/
│   │   ├── architect.md       # System design specialist
│   │   ├── developer.md       # Implementation specialist
│   │   ├── security.md        # Security review specialist
│   │   ├── qa.md              # Testing specialist
│   │   └── coordinator.md     # Workflow orchestrator
│   ├── commands/
│   │   ├── implement.md       # /implement slash command
│   │   └── review.md          # /review slash command
│   └── workflow-state.json    # Persistent workflow state
├── src/                        # Your application code
└── tests/                      # Your test code
```

This entire system lives in your project's `.claude/` directory — version-controlled, portable, and self-contained. No external services, no databases, no MCP servers required.

### Design Principles

| Principle | How It's Enforced |
|-----------|-------------------|
| Role isolation | Each agent's prompt has explicit "NEVER do X" constraints |
| Quality gates | Coordinator checks verdicts before advancing to next step |
| Failure loops | Failed gates route back to the responsible agent, max 3 retries |
| Resumability | `workflow-state.json` tracks progress across sessions |
| Transparency | Each agent reports what it did and what it found |

## Agent Definitions

Each agent is a markdown file with YAML frontmatter. The frontmatter defines metadata; the body is the system prompt.

### Anatomy of an Agent File

```markdown
---
name: architect
description: Designs system architecture. Use when planning a new feature.
model: opus
tools:
  - Read
  - Glob
  - Grep
---

You are the Architect. You design systems.

## Your Role
...

## Rules
- NEVER write application code.
...
```

### Key Design Decisions

**1. Model selection per agent**

Not all agents need the same model:
- `opus` for architect and coordinator — complex reasoning, system design
- `sonnet` for developer, security, qa — fast execution, pattern matching

This optimizes both cost and latency without sacrificing quality where it matters.

**2. Tool scoping**

The `tools:` frontmatter restricts what each agent can access:
- Architect: read-only (Read, Glob, Grep) — cannot modify code
- Developer: read + write (Read, Write, Edit, Bash) — full implementation access
- Security: read-only — can only observe and report
- QA: read + write — can write tests and run them

**3. Explicit boundaries in prose**

Every agent has "NEVER" rules stated in plain language:
- Developer: "NEVER write test code"
- QA: "NEVER modify application code"
- Security: "NEVER write code, only report findings"

These are behavioral constraints, but combined with tool scoping, they create strong guardrails.

## The Implementation Workflow

The `/implement` command triggers the full pipeline:

```mermaid
graph TD
    A[User: /implement feature] --> B[Coordinator]
    B --> C[Step 1: Architect designs]
    C --> D[Step 2: Security reviews design]
    D -->|PASS| E[Step 3: Developer implements]
    D -->|BLOCKED| C
    E --> F[Step 4: Security reviews code]
    F -->|PASS| G[Step 5: QA tests]
    F -->|BLOCKED| E
    G -->|PASS| H[Step 6: Complete]
    G -->|FAIL| E
```

### Quality Gates in Detail

**Gate 1: Design Security Review**
- Security agent reviews the architect's design document
- If CRITICAL/HIGH findings → design goes back to architect for revision
- MEDIUM/LOW findings are passed forward as implementation conditions

**Gate 2: Code Security Review**
- Security agent reviews the implemented code (cold review — never saw the implementation process)
- If CRITICAL/HIGH findings → developer fixes, then security re-reviews
- This catches issues the developer's prompt didn't prevent

**Gate 3: Testing**
- QA writes tests from the *design doc* (not from reading code — prevents mirroring implementation bugs)
- Runs all tests (new + existing)
- If failures → developer gets the bug report, fixes, QA re-runs

### Failure Loop Limits

Each gate allows a maximum of 3 fix cycles. If an issue persists after 3 attempts, the coordinator stops and escalates to the user. This prevents infinite loops when the problem requires human judgment.

## State Management and Resumability

The workflow state is persisted to `.claude/workflow-state.json`:

```json
{
  "feature": "Add due dates to todos",
  "current_step": 3,
  "status": "in_progress",
  "history": [
    {"step": 1, "agent": "architect", "result": "pass", "timestamp": "2026-05-19T10:00:00"},
    {"step": 2, "agent": "security", "result": "pass_with_conditions", "findings": 2, "timestamp": "2026-05-19T10:02:00"},
    {"step": 3, "agent": "developer", "result": "in_progress", "timestamp": "2026-05-19T10:05:00"}
  ]
}
```

If your Claude Code session ends mid-workflow (context limit, timeout, user interruption), the next session can read this state and resume from where it left off.

### How Resumption Works

1. User runs `/implement` again (or coordinator reads state on startup)
2. Coordinator reads `workflow-state.json`
3. If `status: in_progress`, coordinator picks up from `current_step`
4. Previous step outputs (design docs, security findings) are in the filesystem
5. Workflow continues without repeating completed steps

## Comparison with Other Approaches

### vs. Kiro (AWS AI IDE)

Kiro uses a similar pattern (JSON agent definitions + `subagent` tool + workflow skills). Key differences:

| Dimension | Kiro | Claude Code (this pattern) |
|-----------|------|----------------------------|
| Agent format | JSON registry | Markdown files (diffable, readable) |
| Orchestration | Built-in `subagent` tool | Agent tool with coordinator pattern |
| State | Prose tracking files (append-only) | Structured JSON (machine-readable) |
| Gate enforcement | Prompt-behavioral only | Prompt + tool scoping + state checks |
| Portability | Kiro-specific | Any Claude Code environment |
| Parallel execution | Sequential default | `run_in_background` for true parallelism |

### vs. API-Level Orchestration (Python SDK)

The `patterns/agents/orchestrator_workers.ipynb` cookbook shows API-level orchestration in Python. Differences:

| Dimension | Python SDK | Claude Code Native (this pattern) |
|-----------|-----------|------------------------------------|
| Setup | Write Python code | Drop `.claude/` directory into project |
| Invocation | Run Python script | `/implement feature` in CLI |
| Customization | Edit Python classes | Edit markdown files |
| Integration | Standalone script | Lives in your project, uses your codebase |
| State | Application-managed | Filesystem JSON |

The native pattern requires zero code — only markdown configuration. This makes it accessible to any developer, regardless of their SDK expertise.

### vs. Single-Agent Development

| Aspect | Single Agent | Multi-Agent Workflow |
|--------|-------------|---------------------|
| Security review | Self-reviews (biased) | Independent cold review |
| Test quality | Tests mirror implementation | Tests written from design spec |
| Design | Implicit, interleaved with code | Explicit, reviewed before coding |
| Failure detection | After the fact | At each gate (early) |
| Cost | 1 context window | Multiple smaller contexts (can be cheaper) |

## Usage

### Quick Start

1. Copy the `.claude/` directory from this example into your project root
2. Open Claude Code in your project: `claude`
3. Run: `/implement "Add user authentication with JWT"`
4. Watch the agents collaborate through the pipeline

### Available Commands

| Command | What it does |
|---------|-------------|
| `/implement "feature"` | Full pipeline: design → security → code → security → test |
| `/review` | Security + QA review on current uncommitted changes |

### Customization

**Add a new agent** — Create `.claude/agents/your-agent.md` with the frontmatter + prompt format. Update the coordinator's agent table.

**Change the workflow** — Edit the coordinator's "Implementation Workflow" section to add, remove, or reorder steps.

**Adjust model assignments** — Change the `model:` field in any agent's frontmatter. Use `haiku` for mechanical tasks (linting, formatting checks), `sonnet` for implementation, `opus` for complex reasoning.

**Add domain knowledge** — Create `.claude/references/your-standards.md` and instruct agents to read it in their prompts.

### Example Session

```
$ claude
> /implement "Add a priority field (low/medium/high) to todos with filtering"

Coordinator: Starting implementation workflow.

Step 1/6: Routing to Architect...
  Architect: Design document produced. Added priority enum, filter endpoint, sort logic.

Step 2/6: Routing to Security...
  Security: PASS. No findings on design.

Step 3/6: Routing to Developer...
  Developer: Implemented. Modified: src/app.py (+Priority enum, filter method). Created: src/filters.py.

Step 4/6: Routing to Security...
  Security: PASS WITH CONDITIONS. MEDIUM: filter input not validated against enum values.

Step 5/6: Routing to QA...
  QA: 5 new tests written. All 12 tests pass.

Step 6/6: Complete.
  Feature implemented: priority field + filtering.
  Files: src/app.py, src/filters.py, tests/test_filters.py
  Security: cleared (1 MEDIUM finding addressed by developer)
  Tests: 12/12 passing
```

## Design Decisions and Trade-offs

### Why Markdown Files (Not JSON)?

Agent definitions are markdown because:
- **Readable** — anyone can understand what an agent does by reading the file
- **Diffable** — changes to agent behavior show up clearly in git diffs
- **No tooling required** — no build step, no parser, no schema validator
- **Self-documenting** — the prompt IS the documentation

### Why a Coordinator Agent (Not Hardcoded Sequencing)?

The coordinator is an LLM agent, not a script. This means:
- It can make judgment calls (is this output "complete enough"?)
- It can handle unexpected situations (agent reports something outside the workflow)
- It can be overridden by the user mid-workflow

The trade-off: behavioral enforcement is weaker than structural enforcement. The coordinator *could* skip a step. We mitigate this with:
- Explicit "NEVER skip" rules in the prompt
- State file that records every step completion
- The user sees all output and can intervene

### Why Separate Context Windows?

Each agent runs in its own context via the Agent tool. Benefits:
- **Fresh perspective** — security reviewer hasn't been primed by watching the design process
- **Focused context** — each agent's window contains only what it needs
- **Cost efficiency** — smaller contexts use fewer tokens
- **True isolation** — developer cannot accidentally leak into QA's work

### Known Limitations

1. **No structural gate enforcement** — gates are prompt-driven, not code-enforced. A sufficiently confused coordinator could skip a step.
2. **Sequential execution** — agents run one at a time through the coordinator. Parallel execution (e.g., security + QA simultaneously) requires manual orchestration.
3. **Context loss between sessions** — while `workflow-state.json` tracks progress, the detailed context (design discussions, security rationale) lives in agent outputs that may not persist across context resets.
4. **Tool scoping is advisory** — Claude Code's tool permissions are global. The `tools:` frontmatter narrows what is *suggested* to the agent but doesn't hard-block unauthorized tool use.

## Extending the Pattern

### Add a FinOps Agent

For projects with infrastructure costs, add a cost estimation step:

```markdown
# .claude/agents/finops.md
---
name: finops
description: Estimates infrastructure cost impact of changes.
model: haiku
tools:
  - Read
  - Glob
---

You estimate the cost impact of infrastructure changes...
```

### Add Hooks for Audit

Claude Code hooks can enforce structural constraints:

```json
// .claude/settings.json
{
  "hooks": {
    "PostToolUse": [
      {
        "matcher": "Write",
        "command": "echo \"$(date): File written: $FILE_PATH\" >> .claude/audit.log"
      }
    ]
  }
}
```

This creates a structural audit trail — every file write is logged regardless of which agent performed it.

### Batch/Parallel Execution

For multiple features, use Claude Code's headless mode:

```bash
# Process multiple features in parallel
claude -p "/implement Add priority field" &
claude -p "/implement Add due dates" &
wait
```

Each invocation runs its own full workflow in an isolated process.

### Add a Brainstorming Flow

Create `.claude/commands/brainstorm.md` that runs multiple agents in audit mode:
- Security agent: "audit the codebase for vulnerabilities"
- QA agent: "identify untested code paths"
- Architect agent: "identify architectural improvements"
- Coordinator: synthesize and rank findings into a scored backlog

## Summary

This pattern gives you a software engineering team in your `.claude/` directory:

- **Architect** designs before code is written
- **Security** reviews before and after implementation
- **Developer** implements from approved designs only
- **QA** tests from the spec, not the implementation
- **Coordinator** enforces the process and handles failures

### Key Takeaways

1. **Separation of concerns works for AI too.** Narrow prompts produce better output than broad ones.
2. **Quality gates catch issues early.** A security review before implementation is cheaper than a rewrite after.
3. **The pattern is zero-code.** Drop the `.claude/` directory into any project and run `/implement`.
4. **It's composable.** Add agents, add steps, add commands — the pattern scales with your needs.

### Files in This Example

```
multi_agent_dev_workflow/
├── 07_The_multi_agent_dev_workflow.ipynb   # This guide
├── .claude/                                 # The reusable pattern (copy this)
│   ├── CLAUDE.md
│   ├── agents/
│   │   ├── architect.md
│   │   ├── coordinator.md
│   │   ├── developer.md
│   │   ├── qa.md
│   │   └── security.md
│   └── commands/
│       ├── implement.md
│       └── review.md
└── example_project/                         # Minimal project to demo against
    ├── src/app.py
    └── tests/test_app.py
```

### Next Steps

1. Copy `.claude/` into your own project
2. Run `/implement` with a real feature request
3. Read the agents' output — notice how the security review catches things the developer missed
4. Customize: add domain-specific agents, adjust the workflow, tune model assignments
5. Compare the quality of multi-agent output vs. asking a single agent to "build this feature"